# N06 — Lap Time Prediction: XGBoost Model

This notebook trains and evaluates the XGBoost lap time predictor using the feature set
defined in N05.

**Input:**
- `data/processed/laps_featured_2023.parquet` — training set (22,106 laps)
- `data/processed/laps_featured_2024.parquet` — validation set (23,256 laps)
- `data/processed/laps_featured_2025.parquet` — test set (22,760 laps, held-out)
- `data/processed/feature_manifest_laptime.json` — feature registry from N05

**Output:**
- `models/xgb_laptime_v1.json` — trained XGBoost model
- `data/processed/predictions_laptime_val.parquet` — val predictions (2024)
- `notebooks/strategy/lap_time_prediction/outputs/` — evaluation plots

**Model strategy:**
We start with a **single global model** trained on 2023 and evaluated on 2024.
Circuit identity is encoded via `Cluster`, `mean_sector_speed`, and `year_circuit_median` —
the model learns circuit-specific patterns without splitting the data.

> **Contingency:** if per-cluster residual analysis shows systematic underperformance on a
> specific cluster (likely C3 — Monaco / Budapest), we will train one dedicated model per
> cluster as a second iteration.

**Train / Val / Test split:** strictly temporal — no data from 2024 or 2025 is used during
training. No shuffle. This mirrors real deployment conditions where the model predicts
future races from past-season data.


---

## Step 1: Setup & Data Loading

We load the 2023 (train) and 2024 (val) engineered datasets, apply the feature selection
and categorical encoding defined in N05's manifest, and verify the final input shapes
before any modelling.

**Encoding applied here:**
| Feature | Encoding |
|---------|---------|
| `Compound` | Ordinal: SOFT=0, MEDIUM=1, HARD=2, INTERMEDIATE=3, WET=4 |
| `race_phase` | Ordinal: early=0, mid=1, late=2 |
| `FreshTyre` | bool → int |


In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import json
import joblib
import warnings
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [15]:
warnings.filterwarnings("ignore")
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 10


In [16]:
# ── Paths ────────────────────────────────────────────────────────────────────
current_path = Path.cwd()
while not (current_path / ".git").exists() and current_path != current_path.parent:
    current_path = current_path.parent
REPO_ROOT = current_path

PROCESSED = REPO_ROOT / "data" / "processed"
MODELS    = REPO_ROOT / "data" / "models"
OUTPUTS   = REPO_ROOT / "notebooks" / "strategy" / "lap_time_prediction" / "outputs"
MODELS.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

In [17]:
# ── Load manifest ─────────────────────────────────────────────────────────────
manifest = json.load(open(PROCESSED / "feature_manifest_laptime.json"))
TARGET    = manifest['target']
print(f"Target       : {TARGET}")
print(f"Features in  : {manifest['n_features']}")
print(f"Model strategy: {manifest['model_strategy']['approach']}")

Target       : LapTime_s
Features in  : 39
Model strategy: single_global_model


In [18]:
# ── Load parquets ─────────────────────────────────────────────────────────────
df23 = pd.read_parquet(PROCESSED / "laps_featured_2023.parquet")
df24 = pd.read_parquet(PROCESSED / "laps_featured_2024.parquet")

# ── Categorical encoding ──────────────────────────────────────────────────────
COMPOUND_MAP   = manifest['categorical_encoding']['Compound']
RACEPHASE_MAP  = manifest['categorical_encoding']['race_phase']

In [19]:
def add_context_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Recompute session-level context features not stored in the parquet.
    Mirrors N05 Step 4 (anti-drift features), minus the leaky delta_vs_year_circuit_median.
    """
    result = df.copy()

    # Session pace baseline
    result['year_circuit_median'] = (
        result.groupby(['Year', 'GP_Name'])['LapTime_s']
              .transform('median')
    )

    # Constructor ranking by mean lap time per year (rank 1 = fastest)
    team_mean = (
        result.groupby(['Year', 'Team'])['LapTime_s']
              .mean()
              .rename('_team_mean')
              .reset_index()
    )
    team_mean['team_pace_rank'] = (
        team_mean.groupby('Year')['_team_mean']
                 .rank(method='min', ascending=True)
    )
    result = result.merge(
        team_mean[['Year', 'Team', 'team_pace_rank']],
        on=['Year', 'Team'], how='left'
    )
    return result

df23 = add_context_features(df23)
df24 = add_context_features(df24)

print("Context features computed.")
print(f"  year_circuit_median nulls : {df23['year_circuit_median'].isna().sum()}")
print(f"  team_pace_rank nulls      : {df23['team_pace_rank'].isna().sum()}")


Context features computed.
  year_circuit_median nulls : 0
  team_pace_rank nulls      : 0


In [20]:
def encode_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    result['Compound']   = result['Compound'].map(COMPOUND_MAP).fillna(-1).astype(int)
    result['race_phase'] = result['race_phase'].astype(str).map(RACEPHASE_MAP).fillna(-1).astype(int)
    result['FreshTyre']  = result['FreshTyre'].astype(int)
    return result

df23 = encode_features(df23)
df24 = encode_features(df24)

In [21]:
# ── Select features from manifest ─────────────────────────────────────────────
FEATURES = manifest['features_in']

X_train = df23[FEATURES]
y_train = df23[TARGET]

X_val   = df24[FEATURES]
y_val   = df24[TARGET]

print(f"\nTrain (2023) : {X_train.shape[0]:,} laps × {X_train.shape[1]} features")
print(f"Val   (2024) : {X_val.shape[0]:,} laps × {X_val.shape[1]} features")
print(f"\nNull rates train: {X_train.isna().mean().max()*100:.1f}% max")
print(f"Null rates val  : {X_val.isna().mean().max()*100:.1f}% max")
print(f"\nFeature list:")
for i, f in enumerate(FEATURES, 1):
    print(f"  {i:02d}. {f}")


Train (2023) : 22,106 laps × 39 features
Val   (2024) : 23,256 laps × 39 features

Null rates train: 32.9% max
Null rates val  : 33.3% max

Feature list:
  01. DriverNumber
  02. LapNumber
  03. Stint
  04. SpeedI1
  05. SpeedI2
  06. SpeedFL
  07. SpeedST
  08. TyreLife
  09. FreshTyre
  10. Position
  11. CompoundID
  12. TeamID
  13. LapsSincePitStop
  14. FuelLoad
  15. Year
  16. FuelEffect
  17. FuelAdjustedDegPercent
  18. Prev_LapTime
  19. Prev_TyreLife
  20. Prev_SpeedST
  21. LapTime_Delta
  22. SpeedI1_Delta
  23. SpeedI2_Delta
  24. SpeedFL_Delta
  25. SpeedST_Delta
  26. LapTime_Trend
  27. DegradationRate
  28. CumulativeDeg
  29. DegAcceleration
  30. AirTemp
  31. TrackTemp
  32. Humidity
  33. Rainfall
  34. laps_remaining
  35. Cluster
  36. mean_sector_speed
  37. lap_time_pct_of_race_fastest
  38. year_circuit_median
  39. team_pace_rank


### What we see

**39 features** loaded from the manifest — 53 original N04 columns minus 11 pruned in N05
Step 7, plus `year_circuit_median` and `team_pace_rank` computed here.

**Null rates:** max 32.9% (`SpeedI1_Delta`) — expected, documented in N05 Step 1.
XGBoost handles missing values natively via its missing-value branch; no imputation needed.

**Compound and race_phase** are now ordinal integers. `FreshTyre` is 0/1.
All features are numeric — ready for XGBoost.
